In [ ]:
def AttentionUNet(input_shape=(256,256,1), filters=[32,64,128,256]):
    inputs = layers.Input(input_shape)

    # ---- Encoder ----
    c1 = layers.Conv2D(filters[0], 3, activation='relu', padding='same')(inputs)
    c1 = layers.Conv2D(filters[0], 3, activation='relu', padding='same')(c1)
    p1 = layers.MaxPooling2D((2, 2))(c1)

    c2 = layers.Conv2D(filters[1], 3, activation='relu', padding='same')(p1)
    c2 = layers.Conv2D(filters[1], 3, activation='relu', padding='same')(c2)
    p2 = layers.MaxPooling2D((2, 2))(c2)

    c3 = layers.Conv2D(filters[2], 3, activation='relu', padding='same')(p2)
    c3 = layers.Conv2D(filters[2], 3, activation='relu', padding='same')(c3)
    p3 = layers.MaxPooling2D((2, 2))(c3)

    # ---- Bottleneck ----
    bn = layers.Conv2D(filters[3], 3, activation='relu', padding='same')(p3)
    bn = layers.Conv2D(filters[3], 3, activation='relu', padding='same')(bn)

    # ---- Decoder with Attention ----

    # level 3
    u3 = layers.UpSampling2D((2, 2))(bn)
    att3 = attention_gate(c3, u3, inter_channels=filters[2] // 2)
    u3 = layers.Concatenate()([u3, att3])
    c4 = layers.Conv2D(filters[2], 3, activation='relu', padding='same')(u3)
    c4 = layers.Conv2D(filters[2], 3, activation='relu', padding='same')(c4)

    # level 2
    u2 = layers.UpSampling2D((2, 2))(c4)
    att2 = attention_gate(c2, u2, inter_channels=filters[1] // 2)
    u2 = layers.Concatenate()([u2, att2])
    c5 = layers.Conv2D(filters[1], 3, activation='relu', padding='same')(u2)
    c5 = layers.Conv2D(filters[1], 3, activation='relu', padding='same')(c5)

    # level 1
    u1 = layers.UpSampling2D((2, 2))(c5)
    att1 = attention_gate(c1, u1, inter_channels=filters[0] // 2)
    u1 = layers.Concatenate()([u1, att1])
    c6 = layers.Conv2D(filters[0], 3, activation='relu', padding='same')(u1)
    c6 = layers.Conv2D(filters[0], 3, activation='relu', padding='same')(c6)

    outputs = layers.Conv2D(1, (1, 1), activation='sigmoid')(c6)

    model = tf.keras.Model(inputs, outputs)
    return model

# ---- build model ----
att_unet = AttentionUNet()
att_unet.summary()
